# 06 ROCm 调试复盘与排障案例

        这一节把复刻中的折腾整理成排障工作簿。不需要记住所有命令，关键是把一个失败现象拆成证据、根因、修复和验证。


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def find_topic_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError("请从 AMD ROCm 专题目录或 notebooks 子目录启动 Jupyter。")


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
NOTEBOOK_DIR = TOPIC_ROOT / "notebooks"
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/path/to/every-embodied/mujoco_pnp"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/path/to/datasets/every_embodied"))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", PROJECT_ROOT / "ckpt"))

print("TOPIC_ROOT =", TOPIC_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("MODEL_ROOT =", MODEL_ROOT)


In [ ]:
try:
    from IPython.display import Image, Markdown, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

    def Image(filename=None, width=None):
        return f"[image] {filename}"


def show_asset(filename, width=960):
    path = ASSET_DIR / filename
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"缺少素材：{path}")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


## Checkpoint 1：排障记录模板


In [ ]:
template = """
### 问题名

- 现象：
- 复现命令：
- 关键证据：
- 排除项：
- 根因：
- 修复：
- 修复后验证：
- 学习结论：
"""
print(template)


## Checkpoint 2：九个典型案例索引


In [ ]:
rows = [
    ("旧 success 误判", "必须引入 physical_success"),
    ("示教数据 gripper 标签问题", "先审计数据，再怀疑模型"),
    ("ACT loss 下降但闭环失败", "open-loop 与 closed-loop 分开看"),
    ("DAgger 数据混入后退化", "新增数据要说明补的是哪个状态分布"),
    ("SmolVLA 红蓝杯偏置", "按 instruction 拆开评估"),
    ("复制数据伤害另一类任务", "优先试 sampler/loss 层加权"),
    ("视频与 batch 统计不一致", "视频解释行为，成功率看 batch JSONL"),
    ("ROCm 长训练资源问题", "脚本化、分批评估、检查进程和温度"),
    ("pi0 训练前门控", "权限和 smoke 通过后再谈长训练"),
    ("pi0 reset 残留", "同一环境连续评估前先清掉 MuJoCo 动力学状态"),
]
md_table(["案例", "学习结论"], rows)


## Checkpoint 3：pi0 hard-reset 评估协议案例


`dynamic_timed` finisher 的一次排障很典型：stage-aware state 修复后，旧版连续环境评估已经能跑到 strict `30/30`，但 seed `1036` 的终态 `xy` 一度贴近阈值。继续 trace 时发现，后一个 seed 的初始杯子位置偶尔跑出该 seed 的采样范围，说明前一个 episode 的 qvel / ctrl / free-joint 动态状态污染了下一个 episode。

        这里不要急着把它归因成模型泛化问题。更干净的做法是先修评估协议：小面板可以用 `--fresh-env-per-episode` 每个 seed 新建环境；完整批量评估推荐 `--hard-reset-sim-data`，在每次 `env.reset(seed)` 前先清掉底层 MuJoCo sim data。用 hard-reset clean protocol 重跑 unseen seeds `1010-1039` 后，结果仍是 strict `30/30`、legacy `30/30`，mean `xy=0.0219 m`，max `xy=0.0450 m`，seed `1036` 不再贴边。


In [ ]:
rows = [
    ("现象", "旧连续环境评估 30/30，但 seed 1036 一度 max xy=0.0993 m"),
    ("异常证据", "seed 1035 初始杯子位置出现采样范围外的值"),
    ("根因", "env.reset(seed) 重设物体位置，但没有先清掉底层 MuJoCo 动力学残留"),
    ("修复", "新增 --fresh-env-per-episode 和 --hard-reset-sim-data"),
    ("clean 结果", "hard-reset 后 1010-1039 为 strict 30/30，mean xy=0.0219 m，max xy=0.0450 m"),
]
md_table(["项", "记录"], rows)


## Checkpoint 4：把指标和视频放在一起看


In [ ]:
show_asset("act_dagger_progress_curve.png", width=900)
show_asset("smolvla_red_blue_success.png", width=1000)


图表回答“整体表现如何”，视频回答“行为到底像不像成功”。这两类证据最好同时放进实验报告。


## Checkpoint 5：结果摘要模板


In [ ]:
rows = [
    ("设备", "GPU/APU 型号、ROCm、PyTorch、温度和显存"),
    ("数据", "episode 数量、任务类型、是否通过物理回放审计"),
    ("ACT", "best checkpoint、physical_success、主要失败类型"),
    ("SmolVLA", "红杯/蓝杯成功率、采样策略、保护基线"),
    ("pi0", "gated 权限、1-step smoke、正式训练状态"),
    ("视频", "一个真实成功、一个典型失败"),
    ("复盘", "最关键的一个坑和修复证据"),
]
md_table(["项目", "应该写什么"], rows)
